# Supervised Learning: Identifying Crypto Mining Data Centers

The FracTracker data center list does not say which facilities are crypto mines. `02-data-cleaning.ipynb` builds an `is_crypto` label by joining it with the FracTracker crypto tracker (by location and address) and by checking each data center's own `purpose` and `facility_name`.

This notebook loads that label and trains a classifier to predict it.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
PROCESSED_DIR = Path("data/processed")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load data

`02-data-cleaning.ipynb` saves the cleaned data centers and the `is_crypto` label to `data/processed/fractracker_data_centers_clean.csv`. We start from that file.

In [ ]:
data_centers = pd.read_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv")

print("Data centers:", data_centers.shape)
print("Data centers labeled crypto:", data_centers["is_crypto"].sum())

# Train a classifier

Now that every data center has an `is_crypto` label, we can train a model to predict it.

Following the steps from SIADS 542:
1. Pick the features (`X`) and the label (`y`).
2. Split into a training set and a test set.
3. Get a baseline score from a dummy classifier.
4. Train a logistic regression and compare it to the baseline.
5. Look at what the model learned and where it was wrong.

## Pick the features

We kept the features simple:
- `status`: Operating, Proposed, Cancelled, ...
- `sizerank`: the facility's size bucket, from Small to Mega campus, or Unknown.
- `has_sqft`: 1 if FracTracker lists the building's square footage, 0 if not.

`status` and `sizerank` are categories, so we one-hot encode them with `pd.get_dummies`. Each category becomes its own binary variable.

Columns we leave out on purpose to prevent data leakage:
- `purpose` and `facility_name`: part of the label comes from these columns saying "Crypto" or "Bitcoin". Using them would give away the answer (data leakage).
- `lat` and `long`: we used location to make the label, so the model shouldn't use it to predict the label.
- `state`: one-hot encoding would create about 50 extra columns, too many for the dataset with only 59 crypto data centers.

In [ ]:
X = pd.get_dummies(data_centers[["status", "sizerank"]], dtype=int)
X["has_sqft"] = data_centers["facility_size_sqft"].notna().astype(int)

y = data_centers["is_crypto"]

print("X:", X.shape)
print("y:", y.shape)
X.head()

## Split into training and test sets

`train_test_split` holds back 25% of the rows as a test set. The model never sees these rows during training.

Crypto data centers are rare, only about 3.5%. Using `stratify=y` keeps the same proportion of crypto rows in both sets.

We set `random_state=0` for reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

print("Training rows:", len(X_train), "| crypto:", y_train.sum())
print("Test rows:", len(X_test), "| crypto:", y_test.sum())

## Baseline: a dummy classifier

We set up a dummy classifier as a baseline with `strategy="most_frequent"`. This way all our data gets the same prediction, 'not crypto'.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

print("Dummy accuracy:", round(dummy.score(X_test, y_test), 3))

The dummy is about 96.5% accurate without finding a single crypto data center.

This is why accuracy can be misleading when one class is rare. A model can score well just by always guessing the common class. From here on we also look at:
- **Recall** for crypto: of the real crypto data centers, how many did the model find?
- **Precision** for crypto: of the data centers the model called crypto, how many really are?

## Logistic regression: first try

Logistic regression adds up the features, each multiplied by a weight (coefficient), and turns the total into a probability between 0 and 1. If the probability is 0.5 or more, it predicts crypto.

In [ ]:
model = LogisticRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(model.score(X_test, y_test), 3))

# rows = actual (0, 1), columns = predicted (0, 1)
print(confusion_matrix(y_test, predictions))

The model predicts "not crypto" for every row. With about 96.5% of the training rows labeled 0, always guessing 0 is the easiest way for the model to be right most of the time.

## Logistic regression: balanced class weights

Setting `class_weight="balanced"` makes a mistake on a crypto row count more than a mistake on a non-crypto row, because crypto rows are rare. Now ignoring the crypto rows is expensive, so the model has to try to find them.

In [ ]:
model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Accuracy:", round(model.score(X_test, y_test), 3))

print(confusion_matrix(y_test, predictions))
print()
print(classification_report(y_test, predictions))

Accuracy went down, but the model now finds some of the crypto data centers.

In [ ]:
coefficients = pd.Series(model.coef_[0], index=X.columns)
coefficients.sort_values()